# LO5 - LangGraph 워크플로우 설계

LangChain 부품을 흐름(LangGraph)으로 엮는 오케스트레이션을 실습합니다.

## 학습 목표

- StateGraph로 상태, 노드, 엣지를 정의하고 그래프를 컴파일, 실행한다.
- `add_messages` 리듀서로 메시지를 덮어쓰지 않고 누적한다.
- `add_conditional_edges`로 결과에 따라 흐름이 갈라지는 분기를 만든다.
- 순환 그래프에서 `recursion_limit`로 무한 루프를 안전하게 끊는다.

선형 그래프를 먼저 만들고, 같은 그래프에 조건부 엣지를 더해 흐름이 갈라지는 모습을 직접 확인합니다. 마지막으로 순환 그래프에서 `recursion_limit`가 무한 루프를 어떻게 끊는지 눈으로 봅니다.

## 0단계: 패키지 설치 (버전 핀 고정)

버전을 고정해 강의 환경과 동일하게 맞춥니다 (2026-06-04 검증 기준).

In [ ]:
!pip install -U "langchain==1.3.4" "langgraph==1.2.4" langchain-openai langchain-google-genai

## 1단계: API 키 설정과 모델 준비

Colab 좌측 열쇠 아이콘에 키를 한 번 등록하면 이후 모든 실습에서 재사용됩니다. 로컬 환경에서는 환경변수로 키를 설정합니다.

In [ ]:
import os
from langchain.chat_models import init_chat_model

# Colab: 좌측 열쇠 아이콘에 OPENAI_API_KEY를 등록한 뒤 아래 두 줄을 사용합니다.
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# 로컬 대안: 환경변수나 .env에서 키를 읽습니다.
# os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY", "")

# 모델명과 가격은 강의 직전 최신 기준으로 재확인합니다.
model = init_chat_model("openai:gpt-5.4-mini")
# 비용을 줄이려면 대안 모델로 바꿔도 됩니다: init_chat_model("google-genai:gemini-2.5-flash")

## 2단계: 상태 정의와 최소 그래프

`TypedDict`로 상태를 정의하고, 노드 하나를 둔 가장 단순한 선형 그래프를 만듭니다.

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages


class State(TypedDict):
    # messages 키는 덮어쓰지 않고 누적되도록 add_messages 리듀서를 지정합니다.
    messages: Annotated[list, add_messages]


def chatbot(state: State):
    # 현재까지 쌓인 메시지를 모델에 넘기고, 응답을 messages에 추가합니다.
    return {"messages": [model.invoke(state["messages"])]}


builder = StateGraph(State)            # 상태 스키마로 빌더 생성
builder.add_node("chatbot", chatbot)   # 노드 등록
builder.add_edge(START, "chatbot")     # 진입점 연결
builder.add_edge("chatbot", END)       # 종료점 연결
graph = builder.compile()              # 실행 가능한 그래프로 컴파일

# 사용자 메시지로 그래프를 실행합니다.
result = graph.invoke({"messages": [{"role": "user", "content": "LangGraph를 한 문장으로 설명해줘"}]})
print(result["messages"][-1].content)

**체크포인트**

- 반환된 `result["messages"]`에는 사용자 메시지와 모델 응답이 함께 누적돼 있습니다.
- 변형 포인트: `add_messages`를 빼고 `messages: list`로 바꾸면 노드가 돌려준 응답이 입력 메시지를 통째로 덮어써 누적이 깨집니다. 리듀서가 왜 필요한지 직접 비교해 볼 수 있습니다.

## 3단계: 조건부 엣지로 흐름 분기

입력 길이에 따라 처리 흐름을 가르는 라우터를 만듭니다. 긴 입력은 요약 노드를 거치고, 짧은 입력은 바로 종료됩니다.

In [ ]:
# 입력 길이에 따라 처리 흐름을 가르는 라우터를 만듭니다(개념 확인용 분기).
def classify(state: State):
    # classify는 조건부 엣지의 출발 노드가 필요해 둔 패스스루(분류) 노드입니다.
    # 상태를 바꾸지 않고, 실제 분기 판단은 아래 route 라우터 함수가 맡습니다.
    return {"messages": []}  # 빈 리스트를 돌려주므로 상태가 그대로 유지됩니다.


def route(state: State) -> str:
    last = state["messages"][-1].content
    return "summarize" if len(last) > 20 else "end"   # 라우터는 문자열 키를 반환


def summarize(state: State):
    # 긴 입력만 요약 노드를 거칩니다.
    return {"messages": [model.invoke(
        [{"role": "user", "content": f"다음을 한 줄로 요약해줘: {state['messages'][-1].content}"}]
    )]}


b = StateGraph(State)
b.add_node("classify", classify)
b.add_node("summarize", summarize)
b.add_edge(START, "classify")
# 라우터 반환값("summarize"/"end")을 실제 노드 또는 END에 매핑합니다.
b.add_conditional_edges("classify", route, {"summarize": "summarize", "end": END})
b.add_edge("summarize", END)
branch_graph = b.compile()

# 긴 입력은 summarize 노드를 거쳐 모델이 요약한 결과를 출력합니다.
print(branch_graph.invoke({"messages": [{"role": "user", "content": "오늘 회의에서 신제품 출시 일정과 마케팅 예산을 논의했다"}]})["messages"][-1].content)

# 짧은 입력은 route가 "end"를 돌려 summarize 없이 바로 종료합니다.
print(branch_graph.invoke({"messages": [{"role": "user", "content": "안녕"}]})["messages"])

**체크포인트**

- 긴 입력은 `summarize` 노드를 거치고, 짧은 입력은 모델 호출 없이 입력 메시지 그대로 반환됩니다.
- `add_edge`(고정 흐름)와 `add_conditional_edges`(분기)의 차이를 코드로 짚을 수 있습니다.
- 변형 포인트: `route`의 기준 길이(20)를 바꾸면 분기 경계가 이동합니다. 라우터 함수만 고치면 그래프 구조를 건드리지 않고 흐름을 조정할 수 있다는 점이 핵심입니다.

## 4단계: 순환 그래프와 recursion_limit (무한 루프 방지 체험)

종료 조건이 없는 순환 그래프를 일부러 만든 뒤, `recursion_limit`가 무한 루프를 어떻게 끊는지 확인합니다.

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
import operator


class CounterState(TypedDict):
    # 모델 없이 순환만 확인하는 예제라 messages 대신 단순 카운터를 둡니다.
    # operator.add 리듀서로 step 값이 호출마다 누적되도록 합니다.
    step: Annotated[int, operator.add]


def tick(state: CounterState):
    # 매번 step에 1을 더해 돌려줍니다. 종료 조건이 없으면 영원히 반복됩니다.
    return {"step": 1}


def route(state: CounterState) -> str:
    # 의도적으로 종료 조건을 넣지 않아 항상 자기 자신으로 되돌아가게 만듭니다.
    return "loop"  # 실무에서는 여기서 조건을 만족하면 "end"를 돌려줘야 합니다.


b = StateGraph(CounterState)
b.add_node("tick", tick)
b.add_edge(START, "tick")
# tick 다음 라우터가 항상 "loop"를 돌려 tick으로 되돌아오는 순환을 만듭니다.
b.add_conditional_edges("tick", route, {"loop": "tick", "end": END})
loop_graph = b.compile()

# recursion_limit를 5로 낮춰 실행하면, 무한히 도는 대신 한도 초과로 멈춥니다.
try:
    loop_graph.invoke({"step": 0}, {"recursion_limit": 5})
except Exception as err:
    print(type(err).__name__, "발생: 한도를 넘겨 안전하게 중단되었습니다.")

**체크포인트**

- `GraphRecursionError`가 발생하면 한도 초과로 안전하게 멈춘 것입니다.
- 변형 포인트: `route`가 `state["step"] >= 3`일 때 `"end"`를 돌려주도록 바꾸면 세 번 돈 뒤 정상 종료합니다. 종료 조건을 라우터에 넣는 것이 근본 해결책이고, `recursion_limit`는 그 조건이 빠졌을 때 비용 폭주를 막는 안전망입니다.

> ⚠️ **주의**: `recursion_limit`만 믿고 종료 조건을 생략하면 한도까지 매번 모델을 호출해 비용이 그대로 발생합니다. 라우터의 종료 조건이 1순위, `recursion_limit`는 2순위 안전망입니다.

## 실습 체크포인트

- [ ] 0~2단계를 실행해 최소 그래프가 모델 응답을 출력하는지 확인했다
- [ ] `messages`를 `Annotated[list, add_messages]`로 선언했고, 리듀서를 빼면 어떤 문제가 생기는지 설명할 수 있다
- [ ] 3단계에서 긴 입력은 `summarize` 노드를 거치고 짧은 입력은 바로 종료되는 분기를 확인했다
- [ ] `add_edge`와 `add_conditional_edges`의 차이를 코드로 짚을 수 있다
- [ ] 4단계에서 `recursion_limit`로 순환 그래프가 멈추는 것을 확인했고, 근본 해결책이 라우터의 종료 조건임을 설명할 수 있다

## 참고 자료

- [LangGraph 그래프 API](https://docs.langchain.com/oss/python/langgraph/graph-api)
- [LangGraph 영속성](https://docs.langchain.com/oss/python/langgraph/persistence)
- [LangChain 모델](https://docs.langchain.com/oss/python/langchain/models)